In [195]:
import pandas as pd
import numpy as np
import scipy.stats as sps
import geopandas as gpd
import shapefile

In [196]:
# Download walkscore values 
from bs4 import BeautifulSoup
from urllib.request import urlretrieve, urlopen
import xml.etree.ElementTree as ET
import pandas as pd
import requests

In [197]:
url = "https://www.walkscore.com/AU-VIC/Melbourne"

response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

walk_score = []
for q in soup.find_all('tr'):
    line = str(q.text)

    if line[1] == 'R':
        continue

    line = line.split("-")[0].split()[1:]
    score = int(line[-1])
    town = " ".join(line[:-1])
    walk_score.append([town, score])

walk_score_df = pd.DataFrame(walk_score, columns=['town', 'walkability'])
walk_score_df.head()

,town,walkability
0,Carlton,97
1,Fitzroy,96
2,Fitzroy North,93
3,Melbourne,93
4,St Kilda,93


In [198]:
data = pd.read_csv('../data/curated/merged_data.csv')
postcode_to_town = pd.read_csv('../data/curated/postcode_to_town.csv', index_col=0)
data['house'] = np.where(data['property type'] == 'House', 1, 0)
data['apartment'] = np.where(data['property type'] == 'Apartment / Unit / Flat', 1, 0)
data['townhouse'] = np.where(data['property type'] == 'Townhouse', 1, 0)


In [199]:
gb = data.groupby(['postcode']).agg(
    avg_beds=('beds', 'mean'),
    avg_baths=('baths', 'mean'),
    avg_parking=('baths', 'mean'),
    avg_rent=('weekly rent amount', 'mean'),
    avg_per_person_rent=('median weekly household rental payment ($) 2021', 'mean'),
    good_schools=('is closest school good', 'mean'),
    median_rent=('weekly rent amount', 'median'),
    house_percent=('house', 'mean'),
    apartment_percent=('apartment', 'mean'),
    townhouse_percent=('townhouse', 'mean'),
    avg_prmy_school_dist=('distance primary school', 'mean'),
    avg_high_school_dist=('distance secondary school', 'mean'),
    avg_uni_dist=('distance university campus', 'mean'),
    offence_avg=('offence count 2023', 'mean'),
    employed_percent=('employment ratio 2021', 'mean'),
    avg_hospital_dist=('distance to closest hospital', 'mean'),
    population=('estimated resident population (no.) 2021', 'mean'),
    supermarkets=('total_supermarkets', 'mean'),
    coastal=('coastal', 'first'),
    avg_dist_to_city_centre=('closest_cc_dist', 'mean'),
    avg_dist_to_melb=('dist_to_melb', 'mean'),
    avg_dist_to_train=('distance train station', 'mean'),
)

# Now we want to normalise for population
gb['offence_avg'] = gb['offence_avg']
gb['supermarkets'] = gb['supermarkets'] / gb['population']


gb = pd.merge(gb, postcode_to_town, how='left', on='postcode')
gb = pd.merge(gb, walk_score_df, how='left', on='town')

gb_postcode_to_town = postcode_to_town.groupby('postcode')['town'].apply(list)

dct = {}
lst = []

for idx, (postcode, walk) in gb[['postcode', 'walkability']].iterrows():
    dct[postcode] = walk

for idx, (postcode, walk) in gb[['postcode', 'walkability']].iterrows():
    if np.isnan(walk):
        lst.append(dct.get(postcode))
    else:
        lst.append(walk)

gb['walkability'] = lst
gb['walkability'].fillna(np.mean(gb['walkability']), inplace=True)
gb['supermarkets'].fillna(0, inplace=True)

gb['avg_uni_dist'] = np.log(gb['avg_uni_dist'].astype(np.float64) + 0.01)
gb['avg_dist_to_city_centre'] = np.log(gb['avg_dist_to_city_centre'].astype(np.float64) + 1)
gb['avg_dist_to_melb'] = np.log(gb['avg_dist_to_melb'].astype(np.float64) + 1)
gb['avg_hospital_dist'] = np.log(gb['avg_hospital_dist'].astype(np.float64) + 0.01)
gb['avg_prmy_school_dist'] = np.log(gb['avg_prmy_school_dist'].astype(np.float64) + 0.01)
gb['avg_high_school_dist'] = np.log(gb['avg_high_school_dist'].astype(np.float64) + 0.01)
gb['avg_dist_to_train'] = np.log(gb['avg_dist_to_train'].astype(np.float64) + 0.01)

gb.index = gb['postcode']
gb = gb.drop(columns=['town', 'postcode'])
normalised_gb = (gb-gb.min()) / (gb.max()-gb.min())


In [200]:
ldf = pd.DataFrame()
ldf['spaceScore'] = (normalised_gb['avg_beds'] + 0.4*normalised_gb['avg_baths'] + 0.6*normalised_gb['avg_parking']) / 2
ldf['spaceScore'] = np.log(ldf['spaceScore'].astype(np.float64) + 1)
ldf['schoolScore'] = (normalised_gb['good_schools'] + (1 - normalised_gb['avg_prmy_school_dist']) + (1 - normalised_gb['avg_high_school_dist'])) / 3
ldf['uniScore'] = (1 - normalised_gb['avg_uni_dist'])
ldf['affordabilityScore'] = (1 - normalised_gb['median_rent']) + (1 - normalised_gb['avg_rent']) / 2 
ldf['houseScore'] = normalised_gb['house_percent']
ldf['apartmentScore'] = normalised_gb['apartment_percent']
ldf['townhouseScore'] = normalised_gb['townhouse_percent']
ldf['home_typeScore'] = (normalised_gb['house_percent'] + normalised_gb['townhouse_percent'] + 0.5*normalised_gb['apartment_percent']) / 2.5
ldf['safetyScore'] = 1 - normalised_gb['offence_avg']
ldf['employmentScore'] = normalised_gb['employed_percent']
ldf['beachScore'] = normalised_gb['coastal']
ldf['travelScore'] = ((1 - normalised_gb['avg_dist_to_city_centre']) + (1 - normalised_gb['avg_dist_to_train'])) / 2
ldf['melbScore'] = 1 - normalised_gb['avg_dist_to_melb']
ldf['walkability'] = normalised_gb['walkability']
ldf['social_infrastrctureScore'] = (1 - normalised_gb['avg_hospital_dist'])


ldf = (ldf-ldf.min()) / (ldf.max()-ldf.min())
ldf = ldf.drop_duplicates()


In [201]:
# Create a baseline score for each suburb

coefficients = np.array([3, 3, 4, 3, 5, 2, 3, 5, 5, 4]) * 2
#coefficients = []
x = (ldf['spaceScore'], ldf['home_typeScore'], ldf['schoolScore'], ldf['uniScore'], ldf['safetyScore'], ldf['employmentScore'], ldf['travelScore'], ldf['melbScore'], ldf['walkability'], ldf['social_infrastrctureScore'])
ldf['baseline'] = np.dot(coefficients, x)

ranking = pd.merge(ldf, gb_postcode_to_town, on='postcode').sort_values(by='baseline', ascending=False)
ranking = ranking.groupby(['postcode']).agg(baseline=('baseline', 'mean'), town=('town', 'first'))

ranking['baseline'] = 100 * (ranking['baseline'] - ranking['baseline'].min()) / (ranking['baseline'].max() - ranking['baseline'].min())
ranking = ranking.sort_values(by='baseline', ascending=False)

ranking.head(10)

,baseline,town
postcode,,
3002,100.000000,[East Melbourne]
3054,97.480557,"[Carlton North, Princes Hill]"
3065,96.089253,[Fitzroy]
3004,93.920993,[Melbourne]
3141,88.198676,[South Yarra]
3000,86.530421,[Melbourne]
3103,85.264604,"[Balwyn, Balwyn East, Deepdene]"
3070,84.227383,"[Northcote South, Northcote]"
3057,83.415258,[Brunswick East]


In [202]:
# WANT TO HAVE A CHLOROPLETH HERE!! 

shapefile_path = '../data/curated/POSTCODE_POLYGON.shp'


gdf = gpd.read_file(shapefile_path)
gdf.head()

#gdf_merged = gdf.merge(ranking, left_on='postcode_column_in_shapefile', right_on='postcode_column_in_dataframe', how='left')


ShapefileException: Shapefile dbf file must contain at least one field.

In [ ]:

def preferential_postcode(ldf, ranking, safety, school, uni, walkability, house_size, employability, coastal, city, melb):
    """ 
    Creates a scale from 0 to 100, and combines this with the baseline score to find what is important for the user. 
    """
    coefficients = []

    ldf_copy = ldf.copy(deep=True)

    match safety:
        case "Important":
            coefficients.append(3)
        case "Very important":
            coefficients.append(4)
        case "The most important":
            coefficients.append(5)
    
    match school: 
        case "Do not want schools":
            coefficients.append(-5)
        case "Don't care":
            coefficients.append(0)
        case "Important for future":
            coefficients.append(3)
        case "Important now":
            coefficients.append(5)

    match uni:
        case "Do not want unis":
            coefficients.append(-5)
        case "Don't care":
            coefficients.append(0)
        case "Important for future":
            coefficients.append(3)
        case "Important now":
            coefficients.append(5)

    match walkability: 
        case "Not important at all":
            coefficients.append(0)
        case "Not important":
            coefficients.append(1)
        case "Don't care":
            coefficients.append(2)            
        case "Important":
            coefficients.append(4)
        case "The most important":
            coefficients.append(5)

    match house_size:
        case "Apartment":
            coefficients.append(-3)
        case "House, don't mind size":
            coefficients.append(1)
        case "Bigger house":
            coefficients.append(5)

    match employability: 
        case "I'm looking for a job":
            coefficients.append(5)
        case "I'm employed, but worried":
            coefficients.append(3)
        case "I'm employed, but secure":
            coefficients.append(0)    

    if city:
        coefficients.append(5)
    else:
        coefficients.append(-10)

    if melb:
        coefficients.append(5)
    else:
        coefficients.append(-10)

    if coastal:
        ldf_copy = ldf_copy[ldf_copy['beachScore'] == True]

    x = (ldf_copy['safetyScore'], ldf_copy['schoolScore'], ldf_copy['uniScore'],
          ldf_copy['walkability'], ldf_copy['home_typeScore'], ldf_copy['employmentScore'], 
          ldf_copy['travelScore'], ldf_copy['melbScore'])
    ldf_copy['baseline'] = np.dot(coefficients, x)

    pref_ranking = pd.merge(ldf_copy, gb_postcode_to_town, on='postcode').sort_values(by='baseline', ascending=False)
    pref_ranking = pref_ranking.groupby(['postcode']).agg(baseline=('baseline', 'mean'), town=('town', 'first'))

    pref_ranking['baseline'] =  100 * (pref_ranking['baseline'] - pref_ranking['baseline'].min()) / (pref_ranking['baseline'].max() - pref_ranking['baseline'].min())
    pref_ranking = pref_ranking.sort_values(by='baseline', ascending=False)

    end_ranking = pd.merge(pref_ranking, ranking, how='outer', on='postcode', suffixes=['_pref', '_default'])
    end_ranking['lifeScore'] = (end_ranking['baseline_pref'] + end_ranking['baseline_default']) / 2
    end_ranking = end_ranking.drop(columns=['baseline_pref', 'baseline_default', 'town_default']).rename(columns={'town_pref': 'town'}).sort_values(by='lifeScore', ascending=False)


    return end_ranking



In [ ]:
##### UNI STUDENT

# How important is safety to you?
# Important - Very important - The most important 
safety = "Important"
# How important is it to live in a schooling district? 
# Important in future - Important now - Don't care - Do not want schools  
school = "Don't care"
# How important is it to live near a uni?
# Important in future - Important now - Don't care - Do not want unis  
uni = "Important now"
# How important is it to live in a walkable suburb?
# Not important at all - Not important - Don't care - Important - The most important
walkability = "The most important"
# What type of house do you prefer?
# Apartment - House, don't mind size - Bigger house 
house_size = "House, don't mind size"
# Are you worried about employability?
# I'm looking for a job - I'm employed, but worried - I'm employed, but secure
employability = "I'm employed, but worried"
# Do you want to live by the coast?
# Yes - No
coastal = False
# Do you want to live in or near a major city centre?
# Yes - No
city = True
# Do you want to live closer to melbourne?
# Yes - No
melb = True

end_ranking = preferential_postcode(ldf, ranking, safety, school, uni, walkability, house_size, employability, coastal, city, melb)
end_ranking.head(10)

end_ratio = pd.merge(end_ranking, gb['avg_per_person_rent'].drop_duplicates(), how='left', on='postcode')
end_ratio['ratio'] = end_ratio['avg_per_person_rent'] / end_ratio['lifeScore']
end_ratio = end_ratio.sort_values(by='lifeScore', ascending=False)
end_ratio.head(10)

,town,lifeScore,avg_per_person_rent,ratio
postcode,,,,
3002,[East Melbourne],98.527714,480.000000,4.871726
3065,[Fitzroy],96.264763,451.000000,4.684996
3000,[Melbourne],93.265211,412.727273,4.425308
3054,"[Carlton North, Princes Hill]",91.506234,541.000000,5.912165
3004,[Melbourne],89.666304,410.375000,4.576691
3141,[South Yarra],85.891742,426.000000,4.959732
3181,"[Prahran, Prahran East, Windsor]",85.419043,406.500000,4.758892
3006,[Southbank],85.261725,461.000000,5.406881
3003,[West Melbourne],83.847900,385.500000,4.597611


In [ ]:
end_ratio = end_ratio.sort_values(by='ratio', ascending=True)
end_ratio.head(10)

,town,lifeScore,avg_per_person_rent,ratio
postcode,,,,
3000,[Melbourne],93.265211,412.727273,4.425308
3004,[Melbourne],89.666304,410.375000,4.576691
3003,[West Melbourne],83.847900,385.500000,4.597611
3065,[Fitzroy],96.264763,451.000000,4.684996
3181,"[Prahran, Prahran East, Windsor]",85.419043,406.500000,4.758892
3002,[East Melbourne],98.527714,480.000000,4.871726
3141,[South Yarra],85.891742,426.000000,4.959732
3031,"[Flemington, Kensington]",73.662597,380.333333,5.163181
3081,"[Heidelberg Heights, Heidelberg West, Heidelbe...",68.320262,353.000000,5.166842


In [ ]:
##### YOUNG PROFESSIONAL

# How important is safety to you?
# Important - Very important - The most important 
safety = "Very important"
# How important is it to live in a schooling district? 
# Important in future - Important now - Don't care - Do not want schools  
school = "Don't care"
# How important is it to live near a uni?
# Important in future - Important now - Don't care - Do not want unis  
uni = "Don't care"
# How important is it to live in a walkable suburb?
# Not important at all - Not important - Don't care - Important - The most important
walkability = "Important"
# What type of house do you prefer?
# Apartment - House, don't mind size - Bigger house 
house_size = "Apartment"
# Are you worried about employability?
# I'm looking for a job - I'm employed, but worried - I'm employed, but secure
employability = "I'm employed, but secure"
# Do you want to live by the coast?
# Yes - No
coastal = False
# Do you want to live in or near a major city centre?
# Yes - No
city = True
# Do you want to live closer to melbourne?
# Yes - No
melb = True

end_ranking = preferential_postcode(ldf, ranking, safety, school, uni, walkability, house_size, employability, coastal, city, melb)
end_ranking.head(10)

end_ratio = pd.merge(end_ranking, gb['avg_per_person_rent'].drop_duplicates(), how='left', on='postcode')
end_ratio['ratio'] = end_ratio['avg_per_person_rent'] / end_ratio['lifeScore']
end_ratio = end_ratio.sort_values(by='lifeScore', ascending=False)
end_ratio.head(10)

,town,lifeScore,avg_per_person_rent,ratio
postcode,,,,
3002,[East Melbourne],96.229189,480.000000,4.988092
3004,[Melbourne],95.452792,410.375000,4.299246
3000,[Melbourne],93.265211,412.727273,4.425308
3065,[Fitzroy],91.378989,451.000000,4.935489
3141,[South Yarra],87.610287,426.000000,4.862443
3003,[West Melbourne],87.417022,385.500000,4.409896
3054,"[Carlton North, Princes Hill]",86.685510,541.000000,6.240951
3006,[Southbank],85.471622,461.000000,5.393603
3008,[Docklands],83.526352,411.000000,4.920603


In [ ]:
end_ratio = end_ratio.sort_values(by='ratio', ascending=True)
end_ratio.head(10)

,town,lifeScore,avg_per_person_rent,ratio
postcode,,,,
3004,[Melbourne],95.452792,410.375000,4.299246
3003,[West Melbourne],87.417022,385.500000,4.409896
3000,[Melbourne],93.265211,412.727273,4.425308
3141,[South Yarra],87.610287,426.000000,4.862443
3008,[Docklands],83.526352,411.000000,4.920603
3065,[Fitzroy],91.378989,451.000000,4.935489
3002,[East Melbourne],96.229189,480.000000,4.988092
3181,"[Prahran, Prahran East, Windsor]",81.155248,406.500000,5.008918
3051,"[North Melbourne, Hotham Hill]",80.617936,420.666667,5.218028


In [ ]:
##### FAMILY WITH YOUNG KIDS

# How important is safety to you?
# Important - Very important - The most important 
safety = "The most important"
# How important is it to live in a schooling district? 
# Important in future - Important now - Don't care - Do not want schools  
school = "Important now"
# How important is it to live near a uni?
# Important in future - Important now - Don't care - Do not want unis  
uni = "Don't care"
# How important is it to live in a walkable suburb?
# Not important at all - Not important - Don't care - Important - The most important
walkability = "Important"
# What type of house do you prefer?
# Apartment - House, don't mind size - Bigger house 
house_size = "Bigger house"
# Are you worried about employability?
# I'm looking for a job - I'm employed, but worried - I'm employed, but secure
employability = "I'm employed, but secure"
# Do you want to live by the coast?
# Yes - No
coastal = False
# Do you want to live in or near a major city centre?
# Yes - No
city = True
# Do you want to live closer to melbourne cbd?
# Yes - No 
melb = True

end_ranking = preferential_postcode(ldf, ranking, safety, school, uni, walkability, house_size, employability, coastal, city, melb)
end_ranking.head(10)

end_ratio = pd.merge(end_ranking, gb['avg_per_person_rent'].drop_duplicates(), how='left', on='postcode')
end_ratio['ratio'] = end_ratio['avg_per_person_rent'] / end_ratio['lifeScore']
end_ratio = end_ratio.sort_values(by='lifeScore', ascending=False)
end_ratio.head(10)

,town,lifeScore,avg_per_person_rent,ratio
postcode,,,,
3054,"[Carlton North, Princes Hill]",98.740278,541.000000,5.479020
3002,[East Melbourne],97.466617,480.000000,4.924763
3004,[Melbourne],96.256679,410.375000,4.263341
3065,[Fitzroy],93.964087,451.000000,4.799706
3141,[South Yarra],92.553613,426.000000,4.602738
3070,"[Northcote South, Northcote]",88.820821,475.500000,5.353475
3103,"[Balwyn, Balwyn East, Deepdene]",86.952672,507.500000,5.836508
3000,[Melbourne],86.691473,412.727273,4.760875
3057,[Brunswick East],85.770457,434.000000,5.060017


In [ ]:
end_ratio = end_ratio.sort_values(by='ratio', ascending=True)
end_ratio.head(10)

,town,lifeScore,avg_per_person_rent,ratio
postcode,,,,
3004,[Melbourne],96.256679,410.375000,4.263341
3561,"[Nanneella, Rochester, Ballendella]",47.673934,213.000000,4.467850
3141,[South Yarra],92.553613,426.000000,4.602738
3003,[West Melbourne],83.006320,385.500000,4.644225
3260,"[South Purrumbete, Camperdown, Pomborneit, Pom...",50.596483,240.000000,4.743413
3000,[Melbourne],86.691473,412.727273,4.760875
3065,[Fitzroy],93.964087,451.000000,4.799706
3081,"[Heidelberg Heights, Heidelberg West, Heidelbe...",72.911780,353.000000,4.841467
3022,"[Ardeer, Deer Park East]",65.606806,320.000000,4.877543
